# Common Libraries

In [1]:
import os, mujoco, mujoco.viewer
from loop_rate_limiters import RateLimiter
import numpy as np
from mink import (
    Configuration,
    FrameTask,
    PostureTask
)

# Params

In [2]:
myosim_dir_path = "/Users/seojin/myosuite/myosuite/simhive/myo_sim/arm"

# Simulation parameters
duration = 2.0  # seconds
fps = 60
n_frames = int(duration * fps)
dt = 1.0 / fps

solver = "quadprog"
pos_threshold = 1e-4
ori_threshold = 1e-4
max_iters = 20

# Model specification

In [3]:
os.chdir(myosim_dir_path)

model_name = "myoarm.xml"
xml_string = f"""
        <mujoco model="MyoArm with Mocap">
            <include file="{model_name}"/>
            <worldbody>
                <body name="target" pos="0 0 0" quat="0 1 0 0" mocap="true">
                    <geom type="box" size=".15 .15 .15" contype="0" conaffinity="0" rgba=".6 .3 .3 .2"/>
                </body>
            </worldbody>
        </mujoco>
        """

# Initialize model & data

In [4]:
model = mujoco.MjModel.from_xml_string(xml_string)
data = mujoco.MjData(model)

# IK configuration

In [5]:
configuration = Configuration(model)

# Task
tasks = [
    end_effector_task := FrameTask(
        frame_name="S_grasp",
        frame_type="site",
        position_cost=1.0,
        orientation_cost=1.0,
        lm_damping=1.0,
    ),
    posture_task := PostureTask(model=model, cost=1e-2),
]

In [ ]:
# Configuration
configuration = Configuration(model)
origin_target = configuration.get_transform_frame_to_world("S_grasp", "site")

# Renderer
renderer = mujoco.Renderer(configuration.model, height=400, width=600)

# Make target trajectory: move the end-effector outward and then inward in a straight line while keeping the orientation fixed.
start_pos = np.array([0.5, 0.0, 0.4])
end_pos = np.array([0.8, 0.0, 0.4])
orientations = np.array(origin_target.wxyz_xyz[:4])

target_move_outwards = np.linspace(start_pos, end_pos, int(n_frames / 2)) # Move outward
target_move_inwards = np.linspace(end_pos, start_pos, int(n_frames / 2)) # Move inward
target_translations = np.concatenate((target_move_outwards, target_move_inwards), axis = 0) # Move outward and inward
target_orientations = np.tile(orientations, (n_frames,1))
target_move_linearly = np.concatenate((target_orientations, target_translations), axis = 1)

# Make target trajectory: move the end-effector in a circular path.
center_point = np.array([0.3, 0.0, 0.4])
r = 0.05
start_angle = 0
end_angle = 2 * np.pi

angles = np.linspace(start_angle, end_angle, n_frames)
target_translations_circular = np.zeros((n_frames, 3))
target_translations_circular[:, 0] = center_point[0] + r * np.cos(angles)
target_translations_circular[:, 1] = center_point[1] + r * np.sin(angles)
target_translations_circular[:, 2] = center_point[2]

target_orientations = np.tile(orientations, (n_frames, 1))

target_move_circular = np.concatenate((target_orientations, target_translations_circular), axis=1)

# Target Info
target_info = {}

InvalidFrame: site 'trapezium' does not exist in the model. Available site names: ['PECM2_PECM2-P3', 'PECM2_PECM2-P4', 'Thorax_ellipsoid_PECM2_3_sidesite', 'PECM3_PECM3-P3', 'PECM3_PECM3-P4', 'Thorax_ellipsoid_PECM3_3_sidesite', 'LAT1_LAT1-P4', 'LAT1_LAT1-P5', 'LAT1_LAT1-P6', 'LAT2_LAT2-P4', 'LAT2_LAT2-P5', 'LAT2_LAT2-P6', 'LAT3_LAT3-P4', 'LAT3_LAT3-P5', 'LAT3_LAT3-P6', 'C7_marker', 'TMAJ_LAT_PEC_CORBhh_sphere_PECM2_2_sidesite', 'DELT1_DELT1-P4', 'PECM1_PECM1-P3', 'PECM1_PECM1-P4', 'R.Clavicle_marker', 'Deltoid2_sidesite', 'Delt3_sidesite', 'INFSP_sidesite', 'TMIN_sidesite', 'TRIlongglen_sidesite', 's_glenohum_s_glenohum-P2', 'm_glenohum_m_glenohum-P2', 'i_glenohum_i_glenohum-P2', 'coracohum_coracohum-P2', 'DELT1_DELT1-P3', 'DELT2_DELT2-P3', 'DELT2_DELT2-P4', 'Deltoid2_ellipsoid_DELT2_2_sidesite', 'DELT3_DELT3-P1', 'DELT3_DELT3-P2', 'Delt3_torus_DELT3_1_sidesite', 'SUPSP_SUPSP-P2', 'SUPSP_SUPSP-P3', 'INFSP_INFSP-P2', 'INFSP_INFSP-P3', 'SUBSC_SUBSC-P2', 'SUBSC_SUBSC-P3', 'TMIN_TMIN-P2',